# Baseline Solution - Monte Carlo Dropout

### This notebook documents the baseline solution for ADC 2023 without predicting planet radius and temperature

## Overview
Our challenge is to provide a conditional probability distribution for each target. This notebook predicts **5 atmospheric gas abundances only** (log_H2O, log_CO2, log_CO, log_CH4, log_NH3); planet radius and temperature are not predicted. 

Depending on the information content of the observation and the associated observation noise (which is a function of the instrument and the planetary system), the resultant error bounds on each target and their joint conditional distribution will be different.

There are many directions you can take to tackle the problem on hand. We would like to get you started with our baseline solution. Inside this notebook you will find the setup for the baseline model, ways to compute the competition score and how to package the output into the competition format.

Spectroscopic data alone are usually informative enough to provide a reasonable estiamte on the targets. After all, the trough and peaks in the spectra encoded information about the relative abundance of each gaseous species (see [Yip et al.](https://iopscience.iop.org/article/10.3847/1538-3881/ac1744>) ). The supplementary information also helps to better constrain some of the phyiscal quantities (see our discussion [here](https://www.ariel-datachallenge.space/ML/documentation/about) if you want to learn about the underlying physics :) , but I shall leave that to you. 

The baseline solution trains a CNN to output a deterministic estimate for each atmospheric target. At inference time, the network is made to produce probabilistic output by activating the dropout layers in the network (Monte Carlo Dropout, [Gal et al. 2016](https://arxiv.org/abs/1506.02142)). 

In [1]:
import numpy as np
import tensorflow as tf
import pandas as pd
from tensorflow import keras
import h5py
import os
import matplotlib.pyplot as plt
from tqdm import tqdm
from helper import *
from preprocessing import *
from submit_format import to_competition_format
from posterior_utils import *
from spectral_metric import *
from FM_utils_final import *
import taurex.log
taurex.log.disableLogging()
from MCDropout import MC_Convtrainer

ModuleNotFoundError: No module named 'numpy.lib.array_utils'

### Fix seed


In [ ]:
SEED=42

### Constants

In [ ]:
RJUP = 69911000
MJUP = 1.898e27
RSOL = 696340000

## Read training data

In [ ]:
training_path = 'ariel-ml-dataset/TrainingData'


In [ ]:
training_GT_path = os.path.join(training_path, 'Ground Truth Package')

In [ ]:
spectral_training_data = h5py.File(os.path.join(training_path,'SpectralData.hdf5'),"r")
aux_training_data = pd.read_csv(os.path.join(training_path,'AuxillaryTable.csv'))
soft_label_data = pd.read_csv(os.path.join(training_GT_path, 'FM_Parameter_Table.csv'))


## Extract Spectral data
Spectral data lives in a h5py format, which is useful for navigating different cases, but their format makes it difficult to bulk manage them. The helper function helps to transform the h5py file into a matrix of size N x 52 x 4
where N is the number of training examples, 52 is the number of wavelength channels and 4 is the observation data

In [ ]:
spec_matrix = to_observed_matrix(spectral_training_data,aux_training_data)
print("spectral matrix shape:", spec_matrix.shape)

# Visualising a single spectrum

In [ ]:
def visualise_spectrum(spectrum):
    fig = plt.figure(figsize=(10,6))
    ## multiple by 100 to turn it into percentage. 
    plt.errorbar(x=spectrum[:,0], y= spectrum[:,1]*100, yerr=spectrum[:,2]*100 )
    ## we tend to visualise it in log-scale
    plt.xscale('log')
    plt.xlabel('Wavelength (mircon)')
    plt.ylabel('Transit depth (%)')
    plt.show()

In [ ]:
visualise_spectrum(spec_matrix[1])

In [ ]:
## lets look at another one
visualise_spectrum(spec_matrix[2])

it is immediately apparent that the average transit depth between two spectra can change for over an order of magnitude. The magnitude of the uncertainty can also change accordingly ( and is a function of the planetary system, brightness of the host star and instrument response function). 

## Pre-processing

### Settings

In [ ]:
repeat = 5
threshold = 0.8 ## for train valid split.
N = 5000 # train on the first 5000 data instances, remember only some examples are labelled, others are unlabelled!

We can safely discard wlgrid (wavelength grid) and wlwidth (width of wavelength) since they are unchanged in the dataset

### Extract Spectrum

In [ ]:
## extract the noise
noise = spec_matrix[:N,:,2]
## We will incorporate the noise profile into the observed spectrum by treating the noise as Gaussian noise.
spectra = spec_matrix[:N,:,1]
wl_channels = len(spec_matrix[0,:,0])
global_mean = np.mean(spectra)
global_std = np.std(spectra)


### Adding an additional feature - radius of the star 
Most of the time we know something about the planetary system before we even attempt to make an observation (we cant just point randomly with a multi-million euros instrument!). Some of these auxillary data may be useful for retrieval, here we are only using the radius of the star.

In [ ]:
## add Rstar 
Rs = aux_training_data[['star_radius_m',]]
## we would prefer to use Rsol
Rs['star_radius'] = Rs['star_radius_m']/RSOL
Rs = Rs.drop(['star_radius_m'],axis=1)
Rs = Rs.iloc[:N, :]
mean_Rs = Rs.mean()
stdev_Rs = Rs.std()

### Get targets

In [ ]:
# Predict only the 5 gas abundances (no planet_radius or planet_temp)
target_labels = ['log_H2O','log_CO2','log_CO','log_CH4','log_NH3']
targets = soft_label_data.iloc[:N][target_labels]
num_targets = targets.shape[1]
targets_mean = targets.mean()
targets_std = targets.std()

## Train/valid Split

In [ ]:
ind = np.random.rand(len(spectra)) < threshold
training_spectra, training_Rs,training_targets, training_noise = spectra[ind],Rs[ind],targets[ind], noise[ind]
valid_spectra, valid_Rs, valid_targets = spectra[~ind],Rs[~ind],targets[~ind]


## Augment the dataset with noise (create multiple instances)
Observational noise from Ariel forms an important part of the challenge, any model must recognise that the observation are not absolute measurement and could vary (according to the uncertainty), as that will affect the uncertainty associated with our atmospheric targets. Here we try to incorporate these information by augmenting the data with the mean noise.

In [ ]:
aug_spectra = augment_data_with_noise(training_spectra, training_noise, repeat)
aug_Rs = np.tile(training_Rs.values,(repeat,1))
aug_targets = np.tile(training_targets.values,(repeat,1))

### Standardise the data

### spectra

In [ ]:
## standardise the input using global mean and stdev
std_aug_spectra = standardise(aug_spectra, global_mean, global_std)
std_aug_spectra = std_aug_spectra.reshape(-1, wl_channels)
std_valid_spectra = standardise(valid_spectra, global_mean, global_std)
std_valid_spectra = std_valid_spectra.reshape(-1, wl_channels)

### radius

In [ ]:
## standardise
std_aug_Rs= standardise(aug_Rs, mean_Rs.values.reshape(1,-1), stdev_Rs.values.reshape(1,-1))
std_valid_Rs= standardise(valid_Rs, mean_Rs, stdev_Rs)


### target
We are asking the model to provide estimates for 5 atmospheric gas targets (log_H2O, log_CO2, log_CO, log_CH4, log_NH3). In this example we perform a supervised learning task. 

In [ ]:
std_aug_targets = standardise(aug_targets, targets_mean.values.reshape(1,-1), targets_std.values.reshape(1,-1))
std_valid_targets = standardise(valid_targets, targets_mean, targets_std)

# Setup network


### hyperparameter settings


In [ ]:
batch_size= 32
lr= 1e-3
epochs = 30
filters = [32,64,64]
dropout = 0.1
# number of examples to generate in evaluation time (5000 is max for this competition)
N_samples = 5000

We followed [Yip et al.](https://iopscience.iop.org/article/10.3847/1538-3881/ac1744>) and adopted a simple CNN structure and loss function. 


In [ ]:
model = MC_Convtrainer(wl_channels,num_targets,dropout,filters)

### Compile model and Train!

In [ ]:
## compile model and run
model.compile(
    optimizer=keras.optimizers.Adam(lr),
    loss='mse',)
model.fit([std_aug_spectra,std_aug_Rs], 
          std_aug_targets, 
          validation_data=([std_valid_spectra, std_valid_Rs],std_valid_targets),
          batch_size=batch_size, 
          epochs=epochs, 
          shuffle=False,)


In [ ]:
# Save trained CNN weights
model.save_weights("cnn_cnn.weights.h5")  # choose any filename you like

In [ ]:
# Evalute model with validation data

In [ ]:
## select the corresponding GT for the validation data, and in the correct order.
index= np.arange(len(ind))
valid_index = index[~ind]

In [ ]:
instances = N_samples
y_valid_distribution = np.zeros((instances, len(std_valid_spectra), num_targets ))
for i in tqdm(range(instances)):
    
    y_pred_valid = model([std_valid_spectra,std_valid_Rs],training=True)
    y_valid_distribution[i] += y_pred_valid

In [ ]:
y_valid_distribution = y_valid_distribution.reshape(-1,num_targets)

In [ ]:
y_pred_valid_org = transform_and_reshape(
    y_valid_distribution,
    targets_mean.to_numpy(), 
    targets_std.to_numpy(), 
    instances,
    N_testdata=len(std_valid_spectra)
)


In [ ]:
tr1 = y_pred_valid_org
# weight takes into account the importance of each point in the tracedata. for now we just assume them to be equally weighted
weights1 = np.ones((tr1.shape[0],tr1.shape[1]))/np.sum(np.ones(tr1.shape[1]) )


In [ ]:
# now load the ground truth 

In [ ]:
trace_GT = h5py.File(os.path.join(training_GT_path, 'TraceData.hdf5'),"r")

In [ ]:
from sklearn.metrics import mean_squared_error
def visualise_rmse_per_label(y_true, y_pred, labels):
    """
    Computes and visualises the Root Mean Squared Error (RMSE) for each target label.
    
    Args:
        y_true (ndarray or DataFrame): Ground truth values, shape (N_samples, N_targets).
        y_pred (ndarray): Predicted values. Can be shape (N_samples, N_targets) 
                          or (N_samples, N_instances, N_targets) if using MC Dropout.
        labels (list of str): List of target label names.
        
    Returns:
        list: The calculated RMSE scores for each label.
    """
    # Convert y_true to numpy array if it's a pandas DataFrame
    if hasattr(y_true, 'values'):
        y_true = y_true.values
        
    # If y_pred contains multiple MC Dropout instances, average them to get the mean prediction
    if len(y_pred.shape) == 3:
        y_pred = np.mean(y_pred, axis=1)
        
    rmse_scores = []
    
    # Calculate RMSE for each target label
    for i in range(len(labels)):
        mse = mean_squared_error(y_true[:, i], y_pred[:, i])
        rmse_scores.append(np.sqrt(mse))
        
    # Visualisation
    plt.figure(figsize=(10, 6))
    bars = plt.bar(labels, rmse_scores, color='cornflowerblue', edgecolor='black')
    
    # Add the value text on top of each bar
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + (max(rmse_scores)*0.02), 
                 round(yval, 4), ha='center', va='bottom', fontsize=10)
                 
    plt.title('Root Mean Squared Error (RMSE) per Target Label', fontsize=14, pad=15)
    plt.xlabel('Atmospheric Targets', fontsize=12)
    plt.ylabel('RMSE', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
    
    return rmse_scores

In [ ]:
# Execute the visualisation
rmse_results = visualise_rmse_per_label(
    y_true=valid_targets, 
    y_pred=y_pred_valid_org, 
    labels=target_labels
)

In [ ]:
import json
from pathlib import Path

cnn_payload = {
    "rmse": dict(zip(target_labels, [float(x) for x in rmse_results])),
    "rmse_mean": float(np.mean(rmse_results)),
}
Path("cnn_metrics.json").write_text(json.dumps(cnn_payload, indent=2))